# infra-defect-detection — Phase 4 on Kaggle(transformer架构对比:RT-DETR vs. YOLO11n)

**重要:跑这个notebook之前,先在右侧 Session options / Settings 里把 Accelerator 切成 GPU(T4 x2 或 P100)**——默认是CPU-only。

对应路线图阶段4:phase 2在Czech上建立了YOLO11n(CNN)的同国家基线(mAP@50=0.3113)。phase 4要回答Neara那条面试反馈——"缺乏CNN/人脸识别之外的深度学习架构广度"——直接对应的问题:**换成transformer架构(RT-DETR),同样的数据、同样的split、同样的评测方式,表现如何?**

**只改一个变量**:这次训练用的Czech train/val/test split和phase 2完全一样(同样的750/160/162张图,同样的seed=42)——不是重新随机切分,而是用同一个`make_country_split.py --seed 42`重新生成,然后**用一个哈希校验步骤确认和phase 2当时用的split逐张图片完全一致**才往下跑,避免"看起来是同一个split、实际训练数据变了"这种悄悄发生的错误。这样mAP的差异才能干净地归因于"架构换了",而不是"数据也变了"。

## 目录
0. 写入脚本(phase 1/2的5个脚本,不需要phase 3的跨国脚本 + phase 2的CNN基线数字作为对比参照)
1. 安装依赖
2. 下载 + MD5校验(优先用Dataset缓存)
3. 解压转换Czech(优先用Dataset缓存)
4. 重新生成Czech split(seed=42)+ **哈希校验和phase 2的split逐张一致**
5. 训练RT-DETR-L + 在同一个held-out test split上评测 + 自动生成和phase 2的架构对比表
6. 核对结果
7. 收尾:打包成一个zip一次性下载


## 在跑之前:先挂载缓存Dataset(强烈建议)

和phase 3一样,右侧栏 **Add Input** 搜索 `rdd2022-figshare-zip-cache` 加进来,能跳过12.35GB下载 + Czech解压这两步。不加也没关系,只是第2、3步会慢一些。

## 0. 写入脚本 + phase 2 CNN基线数字

In [ ]:
import os
os.makedirs("scripts", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("runs/detect/val", exist_ok=True)


In [ ]:
%%writefile scripts/download_rdd2022.py
"""Download and extract the RDD2022 road damage dataset (roadmap: infra-defect-detection, phase 1).

RDD2022 covers six countries (Japan, India, Czech Republic, Norway, United States, China) with
47,420 road images and 55,000+ annotated damage instances across four classes (D00 longitudinal
crack, D10 transverse crack, D20 alligator crack, D40 pothole). Images are CC BY-SA 4.0 per the
sekilab/RoadDamageDetector GitHub README (attribute sekilab/RoadDamageDetector + the RDD2022 paper,
arxiv.org/abs/2209.08538, in any README/Model Card that uses this data) - note the FigShare listing
below shows "CC BY 4.0" for the same dataset; this discrepancy between the two official sources is
unresolved, so treat CC BY-SA 4.0 (the more restrictive of the two, and the one stated by the
dataset's own authors on their own repo) as the operative license until/unless clarified.

SOURCE CHANGE (2026-09-16): this originally downloaded seven per-country zips from Sekilab's own S3
bucket (bigdatacup.s3.ap-northeast-1.amazonaws.com/.../Country_Specific_Data_CRDDC2022/...). That
bucket now returns HTTP 403 Forbidden on every file, confirmed independently from two unrelated
networks - the bucket's access policy appears to have changed, not a transient fluke. This version
instead downloads FigShare's official combined mirror of the same dataset (one zip, all six
countries, published by the RDD2022/CRDDC2022 organizers themselves), verified by MD5 checksum
against FigShare's own published hash so a partial/corrupted 12GB+ download is caught rather than
silently producing bad data. If FigShare's link ever breaks too, check
https://github.com/sekilab/RoadDamageDetector for current mirror links before assuming this script
is broken.

Because this is one combined zip (not one zip per country), there's no way to download only a
subset of countries - the whole ~12.35GB has to come down regardless. --countries now only controls
which countries get linked into data/raw/ for the conversion step afterward (convert_voc_to_yolo.py
reads data/raw/<country>/), not what gets downloaded.

The zip's exact internal folder layout was not independently verified before writing this script
(Sekilab's own "Directory_Structure_CRDDC_RDD2022.txt" reference file was unreachable from this
environment - see reorganize_countries() below for how this script copes with that uncertainty at
extraction time instead of assuming a fixed layout).

NOTE (2026-09-17, Kaggle handoff): on the original Mac/home-network run, aria2c's multi-connection
mode was confirmed to get an immediate HTTP 403 from FigShare's CDN regardless of connection count
(1, 4, or 16) - the CDN appears to reject Range/segmented requests outright, not just high
concurrency. So on Kaggle this will (correctly) fall back to the plain single-connection urllib
path every time; that's expected, not a bug - the win here is Kaggle's raw single-connection
bandwidth to this host, not multi-connection parallelism.

NOTE (2026-09-17, Kaggle dataset cache): every phase that needs a fresh Kaggle kernel re-downloads
this same ~12.35GB zip from scratch, since /kaggle/working doesn't persist across kernel sessions -
phase 2 confirmed this the hard way. To stop paying that cost every phase, a one-time "cache
builder" kernel downloads + MD5-verifies the zip once and its output becomes a private Kaggle
Dataset; every later notebook just adds that Dataset as an input.

UPDATE (2026-09-17, same day): the cache Dataset did NOT turn out to contain a re-downloadable zip.
Kaggle's "New Dataset from notebook output" flow recursively auto-extracts every .zip file it finds
in the output - not just the outer combined zip, but the 7 per-country zips nested inside it too -
so the Dataset actually ended up holding a fully-extracted, ready-to-convert per-country directory
tree (confirmed by browsing the Dataset's Data Explorer: data/zips/RDD2022_released_through_CRDDC2022/
RDD2022/<Country>/<Country>/{train,test}/... for all 7 countries, ~85.8k files, 13.83GB). That's
actually a BETTER cache than a zip would have been - it skips the extraction step too, not just the
download - so find_kaggle_cached_extracted_countries() below is checked FIRST and, when it covers
every requested country, this script skips straight to "done" (no download, no MD5 verify, nothing -
extract_convert_per_country.py finds and uses the same cache independently). find_kaggle_cached_zip()
is kept as a fallback for a cache Dataset built some other way (e.g. zip auto-extraction turned off),
and the plain FigShare download remains the final fallback - so this script still works unchanged
off-Kaggle, or on a fresh Kaggle account with no cache Dataset set up yet.

Usage:
    python scripts/download_rdd2022.py                        # download + extract + link all found countries
    python scripts/download_rdd2022.py --countries Japan Czech # only link these two after extraction
    python scripts/download_rdd2022.py --skip-extract          # download (+ verify) only
    python scripts/download_rdd2022.py --connections 32        # more aria2c connections (default 16)
"""
import argparse
import hashlib
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

FIGSHARE_URL = "https://ndownloader.figshare.com/files/38030910"
FIGSHARE_ZIP_NAME = "RDD2022_released_through_CRDDC2022.zip"
FIGSHARE_MD5 = "b62bd51d2ffcfaa76c60f234f0cc2bb3"

# Logical country name -> name(s) we'll look for (case-insensitively) among directories inside the
# extracted zip, since the exact internal layout wasn't independently verified (see module docstring).
COUNTRY_ALIASES = {
    "Japan": ["Japan"],
    "India": ["India"],
    "Czech": ["Czech"],
    "Norway": ["Norway"],
    "United_States": ["United_States", "United States", "US", "USA"],
    "China_MotorBike": ["China_MotorBike", "China-MotorBike", "China_Motorbike"],
    "China_Drone": ["China_Drone", "China-Drone"],
}

DEFAULT_DATA_DIR = Path(__file__).resolve().parent.parent / "data"


def _format_bytes(n):
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:.1f}{unit}"
        n /= 1024
    return f"{n:.1f}TB"


def download_with_aria2(url, dest_path, connections=16, retries=3):
    """Segmented, multi-connection download via the aria2c CLI - typically several times faster than
    a single urllib stream on hosts like FigShare's S3-backed CDN, where per-connection throughput
    is often capped well below the link's actual bandwidth. aria2c handles its own retries/resume
    (via its .aria2 control file next to the output), so this just shells out and lets it manage
    that; --continue=true means re-running after an interrupted aria2c download resumes rather than
    restarting from zero.
    """
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        "aria2c",
        "-x", str(connections),          # max connections per server
        "-s", str(connections),          # split file into this many pieces
        "-k", "1M",                      # min split size
        "--continue=true",
        "--max-tries", str(retries),
        "--retry-wait=5",
        "--file-allocation=none",
        "--summary-interval=5",
        # FigShare's CDN (via ndownloader.figshare.com -> a presigned S3 URL) returns 403 to aria2c's
        # default "aria2/x.y.z" User-Agent - matching the header the plain-urllib path already used
        # successfully fixes it. --auto-file-renaming=false avoids aria2 silently writing to a
        # "-1" suffixed file if dest_path.name already exists from an earlier failed attempt.
        "--user-agent=Mozilla/5.0",
        "--auto-file-renaming=false",
        "-d", str(dest_path.parent),
        "-o", dest_path.name,
        url,
    ]
    print(f"  using aria2c with {connections} parallel connections (much faster than a single stream)")
    result = subprocess.run(cmd)
    if result.returncode != 0:
        raise RuntimeError(f"aria2c exited with code {result.returncode} - see its output above for details")


def download_one(url, dest_path, retries=3, connections=16):
    """Download url to dest_path. Skips entirely if dest_path already exists and is non-empty - this
    does NOT check the existing file's checksum, so a corrupted prior download won't be caught here;
    verify_md5() below is what actually guarantees integrity, run separately after this.

    Uses aria2c (multi-connection, much faster) when it's installed on PATH; otherwise falls back to
    a plain single-connection urllib download and prints a one-time hint about installing aria2c for
    a large file like this one. `brew install aria2` on macOS. If aria2c is present but fails (a
    misbehaving CDN, a network that blocks it, etc.), falls back to the urllib path automatically
    rather than giving up outright - not verified against every possible aria2c/network combination,
    so this fallback matters.
    """
    if dest_path.exists() and dest_path.stat().st_size > 0:
        print(f"  already have {dest_path.name} ({_format_bytes(dest_path.stat().st_size)}), skipping download")
        return

    dest_path.parent.mkdir(parents=True, exist_ok=True)

    if shutil.which("aria2c"):
        try:
            download_with_aria2(url, dest_path, connections=connections, retries=retries)
            return
        except RuntimeError as exc:
            print(f"  aria2c failed ({exc}); falling back to a plain single-connection download instead.")
            # clean up whatever partial/zero-byte file aria2c may have left behind before falling back
            if dest_path.exists() and dest_path.stat().st_size == 0:
                dest_path.unlink()
    else:
        print("  NOTE: aria2c not found on PATH - using a single-connection download, which will be "
              "noticeably slower for a file this size. `brew install aria2` (macOS) and re-run for a "
              "multi-connection download instead.")

    _download_urllib(url, dest_path, retries=retries)


def find_kaggle_cached_zip():
    """Look for a pre-cached copy of the combined zip under /kaggle/input/ - a private Kaggle
    Dataset added as this notebook's input, built once by a "cache builder" kernel that just runs
    `download_rdd2022.py --skip-extract` and turns its output into a Dataset (see the module
    docstring's 2026-09-17 note). Kaggle mounts every added dataset read-only at
    /kaggle/input/<dataset-slug>/, so this searches by filename across ALL mounted datasets rather
    than assuming a specific slug - the cache dataset can be renamed, or other unrelated datasets
    can be mounted alongside it, without breaking this. Returns None (not an error) when
    /kaggle/input doesn't exist at all (i.e. not running on Kaggle) or no dataset has the file.
    """
    kaggle_input = Path("/kaggle/input")
    if not kaggle_input.is_dir():
        return None
    matches = list(kaggle_input.rglob(FIGSHARE_ZIP_NAME))
    if not matches:
        return None
    if len(matches) > 1:
        print(f"  NOTE: found {len(matches)} copies of {FIGSHARE_ZIP_NAME} under /kaggle/input/, "
              f"using the first: {matches[0]}")
    return matches[0]


def find_kaggle_cached_extracted_countries(root=None):
    """Look for a pre-EXTRACTED per-country RDD2022 tree under /kaggle/input/ - what the cache
    Dataset actually contains (see the module docstring's 2026-09-17 UPDATE). Kaggle's "New Dataset
    from notebook output" recursively auto-unzips every .zip it finds in the output, including zips
    nested inside other zips - so the cache-builder notebook's output (which only ever contained the
    still-zipped combined archive on disk) turned into a fully-extracted directory tree by the time
    it became a Dataset: both the outer combined zip AND all 7 nested per-country zips got expanded
    in place, not just the outer one. That's a BETTER cache than a re-downloadable zip (skips
    extraction too, not just download), so this is checked before find_kaggle_cached_zip() above.

    Returns {country: path} for every COUNTRY_ALIASES key found as a non-empty directory somewhere
    under root (default /kaggle/input). Does ONE pass over the whole mounted-input tree rather than
    one rglob per country - /kaggle/input can hold 80k+ files once a dataset like this is extracted,
    so scanning it 7x over would be wasteful. Returns {} (not an error) when root doesn't exist (e.g.
    off-Kaggle) or nothing matches. `root` is overridable for self-testing without touching the real
    /kaggle/input.
    """
    kaggle_input = Path(root) if root is not None else Path("/kaggle/input")
    if not kaggle_input.is_dir():
        return {}
    found = {}
    for path in kaggle_input.rglob("*"):
        if len(found) == len(COUNTRY_ALIASES):
            break
        if path.name not in COUNTRY_ALIASES or path.name in found:
            continue
        if not path.is_dir():
            continue
        try:
            if any(path.iterdir()):
                found[path.name] = path
        except OSError:
            continue
    return found


def _download_urllib(url, dest_path, retries=3):
    """Plain single-connection streaming download - the fallback used when aria2c isn't available or
    didn't work."""
    tmp_path = dest_path.with_suffix(dest_path.suffix + ".part")

    for attempt in range(1, retries + 1):
        try:
            print(f"  downloading {url} -> {dest_path} (attempt {attempt}/{retries})")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as resp, open(tmp_path, "wb") as out:
                total = int(resp.headers.get("Content-Length", 0))
                downloaded = 0
                chunk_size = 1024 * 1024
                last_print = time.time()
                while True:
                    chunk = resp.read(chunk_size)
                    if not chunk:
                        break
                    out.write(chunk)
                    downloaded += len(chunk)
                    if time.time() - last_print > 2:
                        pct = f"{downloaded / total:.0%}" if total else "?"
                        print(f"    {_format_bytes(downloaded)}" + (f" / {_format_bytes(total)} ({pct})" if total else ""),
                              end="\r", file=sys.stderr)
                        last_print = time.time()
            tmp_path.rename(dest_path)
            print(f"\n  done: {dest_path.name} ({_format_bytes(dest_path.stat().st_size)})")
            return
        except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, ConnectionError) as exc:
            print(f"\n  attempt {attempt} failed: {exc}")
            if tmp_path.exists():
                tmp_path.unlink()
            if attempt == retries:
                raise
            time.sleep(3 * attempt)


def verify_md5(path, expected_md5):
    """Stream-hash path and compare against expected_md5. Caches a good result next to the file
    (a .md5ok marker) so re-running this script doesn't re-hash a ~12GB file every time - if you
    ever suspect the downloaded file got corrupted after the fact, delete the .md5ok marker (or the
    zip itself) to force a real re-check.
    """
    marker = path.with_suffix(path.suffix + ".md5ok")
    if marker.exists():
        print(f"  {path.name}: MD5 previously verified (delete {marker.name} to re-check)")
        return True

    print(f"  verifying MD5 of {path.name} ({_format_bytes(path.stat().st_size)}, this can take a minute)...")
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(8 * 1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    actual = h.hexdigest()
    if actual != expected_md5:
        print(f"  MD5 MISMATCH: expected {expected_md5}, got {actual}")
        print(f"  {path} is corrupt or incomplete - delete it and re-run this script to re-download.")
        return False
    marker.write_text("ok\n")
    print(f"  MD5 verified: {actual}")
    return True


def extract_one(zip_path, extract_root):
    """Extract zip_path into extract_root, skipping if already extracted (marker-based, not by
    checking for a specific internal folder name - see reorganize_countries() for why)."""
    marker = extract_root / ".extracted"
    if marker.exists():
        print(f"  already extracted into {extract_root}, skipping")
        return
    extract_root.mkdir(parents=True, exist_ok=True)
    print(f"  extracting {zip_path.name} -> {extract_root} (large archive, this can take several minutes)")
    with zipfile.ZipFile(zip_path) as zf:
        bad = zf.testzip()
        if bad is not None:
            raise RuntimeError(f"{zip_path} is corrupt (bad member: {bad}) - delete it and re-run to re-download")
        zf.extractall(extract_root)
    marker.write_text("ok\n")


def reorganize_countries(extract_root, raw_dir, wanted_countries):
    """Find each wanted country's directory somewhere inside the extracted tree and link it to
    raw_dir/<country>, so convert_voc_to_yolo.py (which expects data/raw/<country>/ folders) keeps
    working unchanged regardless of whatever top-level wrapper folder(s) FigShare's zip actually
    uses internally.

    Uses a symlink rather than copying, since the extracted tree is already ~12GB+ and duplicating
    it serves no purpose. For each country, searches for directories whose name matches one of its
    known aliases (case-insensitive) and picks the SHALLOWEST match; if more than one directory at
    that same shallowest depth matches (e.g. the zip ships both a train/ and test/ split each with
    their own per-country subfolder), this prints every candidate found and picks the first
    alphabetically - flagged clearly so you can sanity-check it, rather than silently guessing.
    This is exactly the "raw data doesn't match documentation" messiness this project is meant to
    surface, so warn rather than hide it.
    """
    raw_dir.mkdir(parents=True, exist_ok=True)
    found_any = False

    for country in wanted_countries:
        link_path = raw_dir / country
        if link_path.exists() or link_path.is_symlink():
            print(f"  {country}: {link_path} already exists, leaving as-is")
            found_any = True
            continue

        aliases_lower = {a.lower() for a in COUNTRY_ALIASES[country]}
        candidates = [
            p for p in extract_root.rglob("*")
            if p.is_dir() and p.name.lower() in aliases_lower
        ]
        if not candidates:
            print(f"  WARNING: no directory matching {COUNTRY_ALIASES[country]} found under {extract_root} "
                  f"- {country} will be missing from data/raw/. Check the extracted tree by hand "
                  f"(e.g. `find {extract_root} -iname '*{country.split('_')[0]}*' -type d`) and symlink "
                  f"it manually if this script guessed wrong.")
            continue

        min_depth = min(len(p.relative_to(extract_root).parts) for p in candidates)
        shallowest = sorted(p for p in candidates if len(p.relative_to(extract_root).parts) == min_depth)
        if len(shallowest) > 1:
            print(f"  NOTE: multiple equally-shallow matches for {country}, picking the first:")
            for p in shallowest:
                print(f"    - {p.relative_to(extract_root)}")
        chosen = shallowest[0]
        link_path.symlink_to(chosen, target_is_directory=True)
        print(f"  {country}: linked data/raw/{country} -> {chosen.relative_to(extract_root)}")
        found_any = True

    if not found_any:
        print("  WARNING: none of the requested countries were found - inspect the extracted tree "
              f"under {extract_root} directly; the assumed alias names in COUNTRY_ALIASES may not "
              "match this zip's actual layout.")


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir", default=str(DEFAULT_DATA_DIR),
                         help="root data directory (default: %(default)s)")
    parser.add_argument("--countries", nargs="+", choices=list(COUNTRY_ALIASES) + ["China"], default=None,
                         help="subset of countries to link into data/raw/ after extraction (default: all). "
                              "Does NOT reduce download size - FigShare ships one combined zip for all "
                              "six countries. Pass 'China' for both China_MotorBike and China_Drone.")
    parser.add_argument("--skip-extract", action="store_true", help="download (+ verify) only, don't unzip")
    parser.add_argument("--connections", type=int, default=16,
                         help="parallel connections for aria2c downloads (default: %(default)s, ignored "
                              "if aria2c isn't installed)")
    args = parser.parse_args(argv)

    countries = args.countries
    if countries is None:
        countries = list(COUNTRY_ALIASES)
    elif "China" in countries:
        countries = [c for c in countries if c != "China"] + ["China_MotorBike", "China_Drone"]

    cached_extracted = find_kaggle_cached_extracted_countries()
    missing_from_extracted_cache = [c for c in countries if c not in cached_extracted]
    if cached_extracted and not missing_from_extracted_cache:
        print(f"Found all {len(countries)} requested countries already pre-extracted under "
              f"/kaggle/input/ (see the module docstring's 2026-09-17 UPDATE) - skipping the download "
              f"entirely, no zip needed at all:")
        for country in countries:
            print(f"  {country}: {cached_extracted[country]}")
        print("\nDone (nothing downloaded/extracted here). Next: python scripts/extract_convert_per_country.py "
              "will independently find and use this same cache.")
        return 0
    elif cached_extracted:
        print(f"NOTE: found a partial pre-extracted cache under /kaggle/input/ ({sorted(cached_extracted)}) "
              f"but it's missing {missing_from_extracted_cache} - falling back to the normal "
              f"download/zip-cache path below for all requested countries (not mixing sources).")

    data_dir = Path(args.data_dir)
    zips_dir = data_dir / "zips"
    extract_root = data_dir / "raw" / "_extracted_all"
    raw_dir = data_dir / "raw"

    zip_path = zips_dir / FIGSHARE_ZIP_NAME
    if zip_path.exists() and zip_path.stat().st_size > 0:
        print(f"Already have {zip_path} ({_format_bytes(zip_path.stat().st_size)}), skipping download/cache-copy")
    else:
        cached = find_kaggle_cached_zip()
        if cached:
            print(f"Found a pre-cached copy at {cached} (Kaggle input dataset) - copying into "
                  f"{zip_path} instead of downloading ~12.35GB from FigShare again")
            zips_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(cached, zip_path)
            print(f"  copied {_format_bytes(zip_path.stat().st_size)}")
        else:
            print(f"Downloading combined RDD2022 archive (~12.35GB) from FigShare into {zip_path}")
            download_one(FIGSHARE_URL, zip_path, connections=args.connections)

    if not verify_md5(zip_path, FIGSHARE_MD5):
        print("\nAborting - downloaded file failed MD5 verification. Delete the zip and re-run.")
        return 1

    if args.skip_extract:
        print("\n--skip-extract set, stopping after download+verify.")
        return 0

    print(f"\nExtracting into {extract_root}")
    extract_one(zip_path, extract_root)

    print(f"\nLinking {len(countries)} countries into {raw_dir}")
    reorganize_countries(extract_root, raw_dir, countries)

    print("\nDone. Raw data is under:", raw_dir)
    print("Next: python scripts/convert_voc_to_yolo.py")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scripts/convert_voc_to_yolo.py
"""Convert RDD2022's PASCAL-VOC-XML annotations into YOLO format, and build a unified manifest
across all six countries (roadmap: infra-defect-detection, phase 1).

Why this exists rather than just pointing Ultralytics at the raw VOC XML: (1) YOLO training wants
one .txt label per image with normalized [class cx cy w h] rows, not VOC's per-object XML; (2) more
importantly for this project, we need a single manifest that records which COUNTRY every image
came from, because phase 3's whole point is training on a subset of countries and evaluating
cross-country generalization - that experiment is impossible without country labels surviving the
conversion step.

Design choice - discover files by globbing + matching by filename stem, not by assuming a fixed
"images/" + "annotations/xmls/" subfolder layout: RDD2022's own directory-structure reference file
was unreachable when this project was set up (see download_rdd2022.py's docstring), so hardcoding
an assumed layout would risk silently processing zero files if the real layout differs. Globbing
recursively is slower but correct regardless of how each country's zip is actually organized
internally.

Damage classes (from the RDD2022/CRDDC'2022 label map):
    D00 - longitudinal crack
    D10 - transverse crack
    D20 - alligator crack
    D40 - pothole
Any other class name encountered (older RDD releases had more, e.g. D01/D11/D43/D44/D50) is logged
and SKIPPED, not silently merged into the nearest class - if you see a nontrivial skip count for a
country, that's worth a manual look before assuming the data converted cleanly.

Usage:
    python scripts/convert_voc_to_yolo.py                  # converts every country under data/raw/
    python scripts/convert_voc_to_yolo.py --countries Japan Czech
"""
import argparse
import csv
import sys
import xml.etree.ElementTree as ET
from pathlib import Path

try:
    from PIL import Image
except ImportError:
    Image = None

DEFAULT_DATA_DIR = Path(__file__).resolve().parent.parent / "data"

CLASS_MAP = {"D00": 0, "D10": 1, "D20": 2, "D40": 3}
CLASS_NAMES = ["longitudinal_crack", "transverse_crack", "alligator_crack", "pothole"]


def find_files(root, suffix):
    return sorted(p for p in root.rglob(f"*{suffix}") if p.is_file())


def parse_voc_xml(xml_path):
    """Return (image_filename, width, height, [(class_name, xmin, ymin, xmax, ymax), ...]).

    width/height come from the XML's <size> block when present; if absent or zero (seen in some
    messy real-world VOC exports), the caller falls back to opening the image with PIL - this is
    exactly the kind of "annotation doesn't quite match what a clean benchmark would give you"
    messiness the project is supposed to be diagnosing, so we handle it rather than crash on it.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    filename_el = root.find("filename")
    image_filename = filename_el.text.strip() if filename_el is not None and filename_el.text else xml_path.stem + ".jpg"

    size_el = root.find("size")
    width = height = 0
    if size_el is not None:
        w_el, h_el = size_el.find("width"), size_el.find("height")
        width = int(w_el.text) if w_el is not None and w_el.text else 0
        height = int(h_el.text) if h_el is not None and h_el.text else 0

    objects = []
    for obj in root.findall("object"):
        name_el = obj.find("name")
        bnd = obj.find("bndbox")
        if name_el is None or bnd is None:
            continue
        name = (name_el.text or "").strip()
        try:
            xmin = float(bnd.find("xmin").text)
            ymin = float(bnd.find("ymin").text)
            xmax = float(bnd.find("xmax").text)
            ymax = float(bnd.find("ymax").text)
        except (AttributeError, TypeError, ValueError):
            continue
        objects.append((name, xmin, ymin, xmax, ymax))

    return image_filename, width, height, objects


def voc_box_to_yolo_line(class_idx, xmin, ymin, xmax, ymax, width, height):
    cx = (xmin + xmax) / 2 / width
    cy = (ymin + ymax) / 2 / height
    w = (xmax - xmin) / width
    h = (ymax - ymin) / height
    return f"{class_idx} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


def convert_country(country, raw_dir, processed_dir, manifest_rows, stats):
    xml_files = find_files(raw_dir, ".xml")
    jpg_files = find_files(raw_dir, ".jpg") + find_files(raw_dir, ".jpeg") + find_files(raw_dir, ".JPG")
    image_by_stem = {}
    for p in jpg_files:
        image_by_stem.setdefault(p.stem, p)

    if not xml_files:
        print(f"  WARNING: no .xml annotation files found under {raw_dir} - is the zip actually extracted here?")
        return

    out_images = processed_dir / country / "images"
    out_labels = processed_dir / country / "labels"
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    n_ok = n_orphan_xml = n_bad_size = n_no_objects = n_skipped_class = 0

    for xml_path in xml_files:
        image_filename, width, height, objects = parse_voc_xml(xml_path)
        stem = Path(image_filename).stem

        image_path = image_by_stem.get(stem) or image_by_stem.get(xml_path.stem)
        if image_path is None:
            n_orphan_xml += 1
            continue

        if width <= 0 or height <= 0:
            if Image is None:
                n_bad_size += 1
                continue
            try:
                with Image.open(image_path) as im:
                    width, height = im.size
            except Exception:
                n_bad_size += 1
                continue

        yolo_lines = []
        for name, xmin, ymin, xmax, ymax in objects:
            if name not in CLASS_MAP:
                n_skipped_class += 1
                stats["skipped_classes"][name] = stats["skipped_classes"].get(name, 0) + 1
                continue
            xmin, xmax = sorted((max(0, xmin), min(width, xmax)))
            ymin, ymax = sorted((max(0, ymin), min(height, ymax)))
            if xmax <= xmin or ymax <= ymin:
                continue
            yolo_lines.append(voc_box_to_yolo_line(CLASS_MAP[name], xmin, ymin, xmax, ymax, width, height))
            stats["class_counts"][name] = stats["class_counts"].get(name, 0) + 1

        if not yolo_lines:
            n_no_objects += 1
            continue

        dest_stem = f"{country}__{stem}"
        dest_image = out_images / f"{dest_stem}{image_path.suffix.lower()}"
        dest_label = out_labels / f"{dest_stem}.txt"
        if not dest_image.exists():
            dest_image.write_bytes(image_path.read_bytes())
        dest_label.write_text("\n".join(yolo_lines) + "\n")

        manifest_rows.append({
            "country": country,
            "image": str(dest_image.relative_to(processed_dir.parent)),
            "label": str(dest_label.relative_to(processed_dir.parent)),
            "width": width,
            "height": height,
            "num_objects": len(yolo_lines),
            "classes": ";".join(sorted({l.split()[0] for l in yolo_lines})),
        })
        n_ok += 1

    print(f"  {country}: {n_ok} converted, {n_orphan_xml} orphan xml (no matching image), "
          f"{n_bad_size} bad/unreadable size, {n_no_objects} had zero valid objects after class "
          f"filtering, {n_skipped_class} individual boxes skipped for unrecognized class names")
    stats["per_country"][country] = {
        "converted": n_ok, "orphan_xml": n_orphan_xml, "bad_size": n_bad_size,
        "no_objects": n_no_objects, "skipped_class_boxes": n_skipped_class,
    }


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir", default=str(DEFAULT_DATA_DIR))
    parser.add_argument("--countries", nargs="+", default=None,
                         help="subset of country folder names under data/raw/ to convert (default: all found)")
    args = parser.parse_args(argv)

    if Image is None:
        print("NOTE: Pillow not installed - images with missing/zero <size> in their XML will be "
              "skipped instead of measured. `pip install Pillow` to handle those too.", file=sys.stderr)

    data_dir = Path(args.data_dir)
    raw_dir = data_dir / "raw"
    processed_dir = data_dir / "processed"

    if args.countries:
        countries = args.countries
    else:
        countries = sorted(p.name for p in raw_dir.iterdir() if p.is_dir())

    if not countries:
        print(f"No country folders found under {raw_dir} - run download_rdd2022.py first.", file=sys.stderr)
        return 1

    manifest_rows = []
    stats = {"class_counts": {}, "skipped_classes": {}, "per_country": {}}

    print(f"Converting {len(countries)} countries: {countries}")
    for country in countries:
        print(f"\n[{country}]")
        convert_country(country, raw_dir / country, processed_dir, manifest_rows, stats)

    manifest_path = data_dir / "manifest.csv"
    with open(manifest_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["country", "image", "label", "width", "height", "num_objects", "classes"])
        writer.writeheader()
        writer.writerows(manifest_rows)

    dataset_yaml = data_dir / "dataset.yaml"
    dataset_yaml.write_text(
        "# Auto-generated by convert_voc_to_yolo.py - combined view across all converted countries.\n"
        "# For the phase-3 cross-country experiments, build per-experiment yaml files that point at\n"
        "# a subset of countries' image folders instead of reusing this combined one directly.\n"
        f"path: {processed_dir}\n"
        "train: */images\n"
        f"nc: {len(CLASS_NAMES)}\n"
        f"names: {CLASS_NAMES}\n"
    )

    print(f"\nWrote manifest ({len(manifest_rows)} images) to {manifest_path}")
    print(f"Wrote combined dataset.yaml to {dataset_yaml}")
    print("\nClass distribution across all converted countries:")
    for name, idx in CLASS_MAP.items():
        print(f"  {name}: {stats['class_counts'].get(name, 0)}")
    if stats["skipped_classes"]:
        print("\nSkipped (unrecognized) class names encountered - investigate before trusting counts above:")
        for name, count in sorted(stats["skipped_classes"].items(), key=lambda kv: -kv[1]):
            print(f"  {name!r}: {count}")

    print("\nNext: python scripts/eda_report.py")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scripts/extract_convert_per_country.py
"""Extract + convert RDD2022 ONE COUNTRY AT A TIME, deleting each country's intermediate data as
soon as it's converted (roadmap: infra-defect-detection, phase 1 - Kaggle disk-constrained variant).

WHY THIS EXISTS (2026-09-17, v2->v3->v4): the straightforward approach - download_rdd2022.py's normal
extract_one()/reorganize_countries(), which calls zf.extractall() on the WHOLE combined zip at once
- needs the 12.35GB combined zip AND all the extracted raw VOC data on disk simultaneously, which
blew through a Kaggle notebook's working-directory quota on the first attempt.

v2 tried to fix this by extracting one country's files at a time straight out of the combined zip's
member list - but that assumed the combined zip was a flat tree of images/XML per country. It
ISN'T: FigShare's combined zip (b62bd51d2ffcfaa76c60f234f0cc2bb3, the officially-published MD5) is
actually a ZIP-OF-ZIPS - exactly 7 entries, one per country, each itself a complete nested .zip
(confirmed 2026-09-17 by actually listing the real zip's namelist on Kaggle: `RDD2022/Japan.zip`,
`RDD2022/India.zip`, `RDD2022/Czech.zip`, `RDD2022/Norway.zip`, `RDD2022/United_States.zip`,
`RDD2022/China_MotorBike.zip`, `RDD2022/China_Drone.zip`). v2's matching logic looked for a path
COMPONENT exactly equal to a country alias (e.g. a directory literally named "Japan"), which never
matched because the actual component is "Japan.zip" (a file, not a directory) - so v2 silently
converted 0 images across all 7 countries and wrote an empty manifest.csv.

v3 fixed the matching (match by filename STEM instead of path component) but still copied each
country's nested zip out to a temp file ON DISK before extracting it, while the 12.35GB combined zip
stayed on disk the whole time. That's fine for the small countries, but Norway's nested zip alone is
9.9GB - so by the time v3 reached Norway, disk needed 12.35GB (combined zip, still present, only
deleted at the very end) + already-converted data from the 5 prior countries + a 9.9GB temp copy of
Norway's nested zip, all at once. That blew past Kaggle's 19.5GiB /kaggle/working quota with
`OSError: [Errno 28] No space left on device` mid-copy - not a transient hiccup, this was guaranteed
to happen as soon as processing reached the largest country, regardless of retry.

v4 fixes the actual disk-budget problem instead of just the matching bug:
  1. Read EVERY country's nested-zip bytes into RAM first (`ZipFile.read()`, not `copyfileobj` to a
     temp file) - Kaggle's RAM (~31GB, most of it free) isn't quota-limited the way /kaggle/working
     is, so holding all 7 countries' compressed bytes (summing to the same ~12.35GB as the combined
     zip) in memory costs nothing against the disk quota.
  2. Only ONCE ALL SEVEN have been read into memory - and the combined zip is no longer needed for
     anything - close it and delete the 12.35GB file from disk, BEFORE extracting a single country to
     disk. This frees the full 19.5GiB quota (minus whatever's already used) for the extraction step,
     regardless of which country happens to be biggest or what order they're processed in.
  3. Per country: extract from an in-memory `io.BytesIO` (no on-disk temp zip at all), convert, delete
     the extracted raw folder, and drop that country's bytes from the in-memory dict - so RAM usage
     shrinks back down as we go instead of holding all 7 for the whole run.

v5 (2026-09-17, same day as v4): check for a pre-EXTRACTED per-country cache under /kaggle/input/
FIRST, before any of the v4 zip-reading machinery - see download_rdd2022.py's module docstring
2026-09-17 UPDATE for why the Kaggle Dataset cache ended up holding fully-extracted directories
rather than a re-usable zip (Kaggle auto-extracts zips-within-zips when building a Dataset from
notebook output). When the cache covers every requested country this skips the ENTIRE v4 dance -
convert_country() only ever reads from raw_dir, never writes into it, so it's safe to point it
straight at the read-only /kaggle/input cache path with no copy at all. See
_write_manifest_and_summary() and main() below.

Usage:
    python scripts/extract_convert_per_country.py                  # all countries, then deletes the combined zip
    python scripts/extract_convert_per_country.py --keep-zip       # don't delete the combined zip early or at the end
                                                                    # (only use this if you have >32GB of quota - it
                                                                    # defeats the whole point of the v4 fix above)
    python scripts/extract_convert_per_country.py --data-dir data
"""
import argparse
import csv
import io
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import download_rdd2022 as dl          # noqa: E402  (FIGSHARE_ZIP_NAME, COUNTRY_ALIASES)
import convert_voc_to_yolo as cv       # noqa: E402  (convert_country, CLASS_MAP, CLASS_NAMES)


def _format_bytes(n):
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:.1f}{unit}"
        n /= 1024
    return f"{n:.1f}TB"


DEFAULT_DATA_DIR = Path(__file__).resolve().parent.parent / "data"


def _print_disk_usage(label, working_dir=None):
    result = subprocess.run(["df", "-h", "/"], capture_output=True, text=True)
    print(f"  [{label}] df -h /:\n" + "\n".join("    " + l for l in result.stdout.splitlines()))
    # df -h / reports the container's overall overlay filesystem, which on Kaggle does NOT move in
    # step with /kaggle/working - the thing actually counted against the 19.5GiB quota is disk usage
    # UNDER /kaggle/working specifically, which `du` reports correctly.
    if working_dir is not None:
        du = subprocess.run(["du", "-sh", str(working_dir)], capture_output=True, text=True)
        print(f"  [{label}] du -sh {working_dir}: {du.stdout.strip() or du.stderr.strip()}")


def _write_manifest_and_summary(data_dir, processed_dir, manifest_rows, stats):
    """Shared tail for both the pre-extracted-cache fast path and the full zip-reading slow path in
    main(): write manifest.csv + dataset.yaml and print the class-distribution summary. Factored out
    so the two paths can't silently drift apart on what "done" means."""
    manifest_path = data_dir / "manifest.csv"
    with open(manifest_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["country", "image", "label", "width", "height", "num_objects", "classes"])
        writer.writeheader()
        writer.writerows(manifest_rows)

    dataset_yaml = data_dir / "dataset.yaml"
    dataset_yaml.write_text(
        "# Auto-generated by extract_convert_per_country.py - combined view across all converted countries.\n"
        "# For the phase-3 cross-country experiments, build per-experiment yaml files that point at\n"
        "# a subset of countries' image folders instead of reusing this combined one directly.\n"
        f"path: {processed_dir}\n"
        "train: */images\n"
        f"nc: {len(cv.CLASS_NAMES)}\n"
        f"names: {cv.CLASS_NAMES}\n"
    )

    print(f"\nWrote manifest ({len(manifest_rows)} images) to {manifest_path}")
    print(f"Wrote combined dataset.yaml to {dataset_yaml}")
    print("\nClass distribution across all converted countries:")
    for name in cv.CLASS_MAP:
        print(f"  {name}: {stats['class_counts'].get(name, 0)}")
    if stats["skipped_classes"]:
        print("\nSkipped (unrecognized) class names encountered - investigate before trusting counts above:")
        for name, count in sorted(stats["skipped_classes"].items(), key=lambda kv: -kv[1]):
            print(f"  {name!r}: {count}")

    if not manifest_rows:
        print("\nWARNING: manifest is EMPTY - 0 images were converted. Check the 'Entries matched' listing "
              "above for NOT FOUND countries or unmatched entries before trusting anything downstream.")


def match_country_by_stem(entry_name):
    """Which COUNTRY_ALIASES key this combined-zip entry is, by comparing its filename stem (the
    name with the LAST extension stripped, e.g. "RDD2022/China_MotorBike.zip" -> "China_MotorBike")
    against the known aliases, case-insensitively. This is the v3 fix: v2 matched whole path
    COMPONENTS looking for a directory named e.g. "Japan", which never matched because the real
    entries are files named "Japan.zip", not directories named "Japan"."""
    stem_lower = Path(entry_name).stem.lower()
    for country, aliases in dl.COUNTRY_ALIASES.items():
        for alias in aliases:
            if alias.lower() == stem_lower:
                return country
    return None


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir", default=str(DEFAULT_DATA_DIR))
    parser.add_argument("--keep-zip", action="store_true",
                         help="don't delete the combined zip early (once all countries are read into "
                              "memory) or at the end - default is to delete it early, which is what "
                              "makes the largest country (Norway, ~9.9GB) fit under Kaggle's quota")
    parser.add_argument("--countries", nargs="+", default=None,
                         help="only extract+convert these countries (default: all 7). Added for phase 2, "
                              "which only needs a single country (e.g. --countries Czech) - still reads "
                              "the whole combined zip's member list, but only pulls the requested "
                              "countries' bytes into memory, so the early-delete-the-zip step still runs "
                              "and disk usage stays tiny.")
    args = parser.parse_args(argv)

    data_dir = Path(args.data_dir)
    zip_path = data_dir / "zips" / dl.FIGSHARE_ZIP_NAME
    raw_dir = data_dir / "raw"
    processed_dir = data_dir / "processed"
    working_dir = data_dir.resolve().parent  # e.g. /kaggle/working - what the quota actually tracks

    processed_dir.mkdir(parents=True, exist_ok=True)
    manifest_rows = []
    stats = {"class_counts": {}, "skipped_classes": {}, "per_country": {}}

    # v5 (2026-09-17, same day as v4 above): check for a pre-EXTRACTED per-country cache under
    # /kaggle/input/ first - see download_rdd2022.py's module docstring 2026-09-17 UPDATE for why the
    # cache Dataset ended up holding fully-extracted directories rather than a re-usable zip (Kaggle
    # auto-extracts zips-within-zips when building a Dataset from notebook output). When it covers
    # every requested country this skips the ENTIRE zip-reading/in-memory-extraction dance above -
    # convert_country() only ever reads from raw_dir (rglob for .xml/.jpg, then read_bytes()), never
    # writes into it, so it's safe to point it straight at the read-only /kaggle/input cache path with
    # no copy at all.
    requested_countries = args.countries if args.countries else list(dl.COUNTRY_ALIASES)
    cached_dirs = dl.find_kaggle_cached_extracted_countries()
    missing_from_cache = [c for c in requested_countries if c not in cached_dirs]

    if cached_dirs and not missing_from_cache:
        print(f"Found all {len(requested_countries)} requested countries pre-extracted under "
              f"/kaggle/input/ (a Kaggle Dataset cache) - converting straight from there, no zip "
              f"involved at all:")
        for country in requested_countries:
            print(f"  {country}: {cached_dirs[country]}")
        for country in requested_countries:
            print(f"\n[{country}] converting directly from the cached extracted tree...")
            cv.convert_country(country, cached_dirs[country], processed_dir, manifest_rows, stats)
        _write_manifest_and_summary(data_dir, processed_dir, manifest_rows, stats)
        print("\nNext: python scripts/eda_report.py")
        return 0
    elif cached_dirs:
        print(f"NOTE: found a partial pre-extracted cache under /kaggle/input/ ({sorted(cached_dirs)}) "
              f"but it's missing {missing_from_cache} - falling back to the full combined-zip path "
              f"below for ALL requested countries (not mixing sources, to keep this predictable).")

    if not zip_path.exists():
        print(f"{zip_path} not found - run download_rdd2022.py --skip-extract first.", file=sys.stderr)
        return 1

    # Clean slate for raw_dir: it's always transient intermediate storage, never a final deliverable,
    # so wipe any leftover partial state from a previous crashed run (e.g. a half-written
    # _nested_Norway.zip from the v3 run that hit the disk-quota error) before starting.
    if raw_dir.exists():
        print(f"Removing leftover {raw_dir} from a previous run (transient data only, safe to wipe)...")
        shutil.rmtree(raw_dir)
    raw_dir.mkdir(parents=True, exist_ok=True)

    print(f"Opening {zip_path} to read its member list (no extraction yet, costs no disk space)...")
    outer_zf = zipfile.ZipFile(zip_path)
    infos = [i for i in outer_zf.infolist() if not i.filename.endswith("/")]
    print(f"  combined zip contains {len(infos)} entries")

    by_country = {}
    unmatched = []
    for info in infos:
        country = match_country_by_stem(info.filename)
        if country:
            by_country[country] = info
        else:
            unmatched.append(info.filename)

    print("\nEntries matched (this is the combined zip's REAL layout - one nested zip per country):")
    for country in dl.COUNTRY_ALIASES:
        if country in by_country:
            info = by_country[country]
            print(f"  {country}: {info.filename} ({_format_bytes(info.file_size)})")
        else:
            print(f"  {country}: NOT FOUND")

    if args.countries:
        missing = [c for c in args.countries if c not in by_country]
        if missing:
            print(f"\n--countries requested {missing} but the combined zip doesn't have (a match for) "
                  f"{'them' if len(missing) > 1 else 'it'} - check spelling against COUNTRY_ALIASES.", file=sys.stderr)
            return 1
        by_country = {c: by_country[c] for c in args.countries}
        print(f"\n--countries set: only extracting {list(by_country.keys())} (out of all 7 found above)")

    if unmatched:
        print(f"\n  {len(unmatched)} entries matched no known country alias:")
        for n in unmatched:
            print(f"    {n}")

    _print_disk_usage("before reading anything", working_dir)

    # Step 1: read EVERY country's nested-zip bytes into RAM before extracting any of them to disk.
    # This is what lets us delete the 12.35GB combined zip BEFORE the largest country (Norway,
    # 9.9GB) needs to be extracted, instead of only being able to delete it at the very end.
    print(f"\nReading all {len(by_country)} countries' nested-zip bytes into memory (RAM isn't "
          f"quota-limited the way /kaggle/working is - this avoids ever needing the 12.35GB combined "
          f"zip and a country's extracted data on disk at the same time)...")
    country_bytes = {}
    for country, info in by_country.items():
        print(f"  reading {country} ({info.filename}, {_format_bytes(info.file_size)}) into memory...")
        country_bytes[country] = outer_zf.read(info)
    zip_size = zip_path.stat().st_size
    outer_zf.close()

    if not args.keep_zip:
        zip_path.unlink()
        print(f"\nDeleted {zip_path} ({_format_bytes(zip_size)}) early - every country's bytes are "
              f"already in memory, so the combined zip is no longer needed and this frees up disk "
              f"headroom before extracting any country.")
    else:
        print(f"\n--keep-zip set: leaving {zip_path} ({_format_bytes(zip_size)}) on disk (less headroom "
              f"for the extraction step below - only safe with a much larger quota than 19.5GiB).")
    _print_disk_usage("after reading all countries into memory" + ("" if args.keep_zip else " + deleting the combined zip"), working_dir)

    # Step 2: per country, extract from the in-memory bytes (no on-disk temp zip), convert, clean up.
    for country in list(country_bytes.keys()):
        data = country_bytes.pop(country)  # drop from the dict now so RAM shrinks as we go

        country_raw = raw_dir / country
        if country_raw.exists():
            shutil.rmtree(country_raw)
        country_raw.mkdir(parents=True)

        print(f"\n[{country}] extracting {_format_bytes(len(data))} (in memory) into {country_raw}...")
        with zipfile.ZipFile(io.BytesIO(data)) as inner_zf:
            bad = inner_zf.testzip()
            if bad is not None:
                print(f"  WARNING: {country}'s nested zip is corrupt (bad member: {bad}) - skipping {country}")
                shutil.rmtree(country_raw)
                del data
                continue
            inner_zf.extractall(country_raw)
        del data  # free this country's RAM now that it's on disk as extracted files

        print(f"  converting...")
        cv.convert_country(country, country_raw, processed_dir, manifest_rows, stats)

        print(f"  deleting {country_raw} (already converted, no longer needed) to free space for the next country...")
        shutil.rmtree(country_raw)
        _print_disk_usage(f"after {country}", working_dir)

    _write_manifest_and_summary(data_dir, processed_dir, manifest_rows, stats)
    _print_disk_usage("final", working_dir)
    print("\nNext: python scripts/eda_report.py")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scripts/make_country_split.py
"""Build a reproducible train/val/test split for a single country's data, from manifest.csv
(roadmap: infra-defect-detection, phase 2 - CNN baseline).

Why a separate script rather than letting Ultralytics auto-split: phase 2's whole point is to
establish a "same-country train/test" baseline number that phase 3 later compares a cross-country
number against - that comparison is only meaningful if the split is fixed, recorded, and re-usable
(so phase 3's report can say "the same Czech test set" rather than "some random subset"). Ultralytics
can auto-split a folder, but doesn't write out a reproducible, inspectable record of which images
ended up in which split - so we do it ourselves and hand Ultralytics explicit image-list .txt files.

Usage:
    python scripts/make_country_split.py --country Czech
    python scripts/make_country_split.py --country Czech --train-frac 0.7 --val-frac 0.15 --seed 42
"""
import argparse
import csv
import json
import random
import sys
from pathlib import Path

DEFAULT_DATA_DIR = Path(__file__).resolve().parent.parent / "data"

CLASS_NAMES = ["longitudinal_crack", "transverse_crack", "alligator_crack", "pothole"]


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--country", required=True, help="country name as it appears in manifest.csv's country column")
    parser.add_argument("--data-dir", default=str(DEFAULT_DATA_DIR))
    parser.add_argument("--train-frac", type=float, default=0.7)
    parser.add_argument("--val-frac", type=float, default=0.15)
    parser.add_argument("--seed", type=int, default=42, help="fixed shuffle seed - record this alongside any reported metric")
    args = parser.parse_args(argv)

    if args.train_frac + args.val_frac >= 1.0:
        print(f"--train-frac ({args.train_frac}) + --val-frac ({args.val_frac}) must leave room for a "
              f"non-empty test split (currently sums to {args.train_frac + args.val_frac})", file=sys.stderr)
        return 1

    data_dir = Path(args.data_dir)
    manifest_path = data_dir / "manifest.csv"
    if not manifest_path.exists():
        print(f"{manifest_path} not found - run extract_convert_per_country.py first.", file=sys.stderr)
        return 1

    with open(manifest_path, newline="") as f:
        rows = [r for r in csv.DictReader(f) if r["country"] == args.country]

    if not rows:
        print(f"No rows for country={args.country!r} in {manifest_path} - check spelling, or that "
              f"extract_convert_per_country.py was run with --countries {args.country}.", file=sys.stderr)
        return 1

    # Sort first so the shuffle is deterministic regardless of manifest.csv's row order (which
    # itself depends on filesystem iteration order and isn't guaranteed stable across re-runs).
    rows.sort(key=lambda r: r["image"])
    rng = random.Random(args.seed)
    rng.shuffle(rows)

    n = len(rows)
    n_train = int(n * args.train_frac)
    n_val = int(n * args.val_frac)
    splits = {
        "train": rows[:n_train],
        "val": rows[n_train:n_train + n_val],
        "test": rows[n_train + n_val:],
    }

    split_dir = data_dir / "splits" / args.country
    split_dir.mkdir(parents=True, exist_ok=True)

    for split_name, split_rows in splits.items():
        list_path = split_dir / f"{split_name}.txt"
        with open(list_path, "w") as f:
            for r in split_rows:
                # manifest.csv's "image" column is already relative to data_dir (e.g.
                # "processed/Czech/images/Czech__xxx.jpg") - Ultralytics wants absolute paths to be
                # safe regardless of what directory `yolo train` is invoked from.
                f.write(str((data_dir / r["image"]).resolve()) + "\n")
        print(f"  {split_name}: {len(split_rows)} images -> {list_path}")

    if any(len(v) == 0 for v in splits.values()):
        print(f"\nWARNING: at least one split is empty (train={len(splits['train'])}, "
              f"val={len(splits['val'])}, test={len(splits['test'])}) - {args.country} may not have "
              f"enough images for these fractions.", file=sys.stderr)

    dataset_yaml_path = split_dir / "dataset.yaml"
    dataset_yaml_path.write_text(
        f"# Auto-generated by make_country_split.py for country={args.country}, seed={args.seed}.\n"
        f"# Labels are found by Ultralytics' convention: same path with the LAST 'images' path "
        f"component replaced by 'labels' and the extension replaced by .txt - this matches how "
        f"convert_voc_to_yolo.py laid out data/processed/{args.country}/images/ + labels/.\n"
        f"train: {(split_dir / 'train.txt').resolve()}\n"
        f"val: {(split_dir / 'val.txt').resolve()}\n"
        f"test: {(split_dir / 'test.txt').resolve()}\n"
        f"nc: {len(CLASS_NAMES)}\n"
        f"names: {CLASS_NAMES}\n"
    )
    print(f"  dataset.yaml -> {dataset_yaml_path}")

    split_config = {
        "country": args.country,
        "seed": args.seed,
        "train_frac": args.train_frac,
        "val_frac": args.val_frac,
        "test_frac": round(1.0 - args.train_frac - args.val_frac, 6),
        "n_total": n,
        "n_train": len(splits["train"]),
        "n_val": len(splits["val"]),
        "n_test": len(splits["test"]),
    }
    config_path = split_dir / "split_config.json"
    config_path.write_text(json.dumps(split_config, indent=2) + "\n")
    print(f"  split_config.json -> {config_path}")
    print(f"\n{args.country}: {n} total images -> train={len(splits['train'])} "
          f"val={len(splits['val'])} test={len(splits['test'])} (seed={args.seed})")
    print(f"\nNext: python scripts/train_cnn_baseline.py --data {dataset_yaml_path}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scripts/train_transformer_baseline.py
"""Train + evaluate a single-country RT-DETR (transformer) baseline on the SAME split as phase 2's
YOLO11 (CNN) baseline (roadmap: infra-defect-detection, phase 4 - architecture comparison).

WHY THIS EXISTS: phase 2 established a same-country CNN baseline (YOLO11n on Czech, mAP@50=0.3113).
The interview feedback this project responds to (Neara) specifically flagged a lack of evidence of
architecture breadth beyond CNN/face-recognition work - phase 4 closes that gap by fine-tuning a
transformer detector (RT-DETR, Ultralytics' real-time DETR variant) and comparing it against the
CNN baseline with EXACTLY ONE variable changed: the model architecture. Everything else - the
Czech train/val/test split (same 750/160/162 images, same seed=42), the held-out-test evaluation
discipline, the metrics reported - is held constant, because a comparison that also changes the
data isn't a clean architecture comparison (that's why this script does NOT regenerate the split
via make_country_split.py: it expects the exact split files phase 2 already produced and committed
under data/splits/Czech/, so both runs are trained/evaluated on byte-identical images).

Usage:
    python scripts/train_transformer_baseline.py --data data/splits/Czech/dataset.yaml
    python scripts/train_transformer_baseline.py --data data/splits/Czech/dataset.yaml \\
        --compare-against runs/detect/val/baseline_metrics.json
"""
import argparse
import json
import sys
import time
from pathlib import Path

import yaml
from ultralytics import RTDETR

DEFAULT_RUNS_DIR = Path(__file__).resolve().parent.parent / "runs" / "phase4"
DEFAULT_CNN_METRICS = Path(__file__).resolve().parent.parent / "runs" / "detect" / "val" / "baseline_metrics.json"


def _load_cnn_comparison(path):
    """Load phase 2's CNN baseline_metrics.json for the comparison table, if it's present. Returns
    None (not an error) when the file is missing - the report just skips the comparison section
    rather than failing, since this script is still useful standalone (e.g. re-running with a
    different seed/model) even without phase 2's numbers on hand."""
    if path is None or not Path(path).exists():
        return None
    return json.loads(Path(path).read_text())


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data", required=True, help="path to the SAME dataset.yaml phase 2 used (data/splits/Czech/dataset.yaml)")
    parser.add_argument("--model", default="rtdetr-l.pt",
                         help="Ultralytics RT-DETR checkpoint to start from (default: rtdetr-l.pt, "
                              "COCO-pretrained - RT-DETR-L, not the larger -x variant, to keep "
                              "training time controlled on a single Kaggle GPU). Use a bare "
                              "'rtdetr-l.yaml' instead to train from random init with no internet "
                              "download, e.g. if the pretrained-weights download is ever unavailable.")
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument("--batch", type=int, default=8,
                         help="default 8, not phase 2's 16 - RT-DETR-L (~32M params) has a "
                              "noticeably larger memory footprint than YOLO11n (~2.6M params) at "
                              "the same imgsz, so this halves the batch size to fit Kaggle's T4/P100 "
                              "16GB headroom. Raise it if you confirm your GPU has room.")
    parser.add_argument("--seed", type=int, default=42, help="must match the seed already baked into the split files (42)")
    parser.add_argument("--patience", type=int, default=20, help="early-stop patience (epochs with no val improvement)")
    parser.add_argument("--project", default=str(DEFAULT_RUNS_DIR))
    parser.add_argument("--name", default=None, help="run name (default: derived from the country in --data)")
    parser.add_argument("--compare-against", default=str(DEFAULT_CNN_METRICS),
                         help="path to phase 2's baseline_metrics.json (CNN/YOLO11n numbers) to build "
                              "a Δ comparison table against - default is where phase 2 actually wrote "
                              "it. Pass an empty string to skip the comparison section entirely.")
    args = parser.parse_args(argv)

    data_yaml_path = Path(args.data)
    if not data_yaml_path.exists():
        print(f"{data_yaml_path} not found - this script expects phase 2's already-committed split "
              f"(data/splits/Czech/dataset.yaml), not a fresh make_country_split.py run.", file=sys.stderr)
        return 1

    with open(data_yaml_path) as f:
        data_cfg = yaml.safe_load(f)
    class_names = data_cfg.get("names", [])

    split_config_path = data_yaml_path.parent / "split_config.json"
    split_config = json.loads(split_config_path.read_text()) if split_config_path.exists() else None
    country = (split_config or {}).get("country") or data_yaml_path.parent.name
    run_name = args.name or f"{country.lower()}_rtdetr"

    cnn_metrics = _load_cnn_comparison(args.compare_against) if args.compare_against else None
    if args.compare_against and cnn_metrics is None:
        print(f"NOTE: --compare-against {args.compare_against!r} not found - proceeding without a "
              f"CNN comparison table (this run's own numbers are still written normally).")
    elif cnn_metrics is not None:
        if split_config and cnn_metrics.get("split_config") and cnn_metrics["split_config"] != split_config:
            print(f"WARNING: the CNN comparison file's split_config does not match this run's "
                  f"split_config.json - the two runs may not have used the same train/val/test "
                  f"images, which would invalidate a direct comparison. Proceeding anyway, but check "
                  f"this before trusting the Δ numbers.", file=sys.stderr)

    print(f"Training {args.model} on {country} (data={data_yaml_path}, seed={args.seed}, "
          f"epochs={args.epochs}, imgsz={args.imgsz}, batch={args.batch})...")
    if split_config:
        print(f"  split (from {split_config_path.name}): train={split_config['n_train']} "
              f"val={split_config['n_val']} test={split_config['n_test']} "
              f"(split seed={split_config['seed']}, fractions "
              f"{split_config['train_frac']}/{split_config['val_frac']}/{split_config['test_frac']})")

    model = RTDETR(args.model)
    t0 = time.time()
    model.train(
        data=str(data_yaml_path),
        epochs=args.epochs,
        imgsz=args.imgsz,
        batch=args.batch,
        seed=args.seed,
        patience=args.patience,
        project=args.project,
        name=run_name,
        exist_ok=True,
        plots=True,
    )
    train_seconds = time.time() - t0

    # Evaluate on the held-out TEST split, same discipline as phase 2 - never seen during training or
    # checkpoint selection. Explicit project/name (unlike letting Ultralytics default to runs/detect/val)
    # so this doesn't collide with/overwrite phase 2's CNN val run living at that same default path.
    print(f"\nEvaluating best checkpoint on the TEST split (never seen during training)...")
    eval_project = str(DEFAULT_RUNS_DIR / "eval")
    test_results = model.val(data=str(data_yaml_path), split="test", plots=True,
                              project=eval_project, name=run_name, exist_ok=True)
    run_dir = Path(test_results.save_dir)

    box = test_results.box
    per_class = []
    for idx in range(len(class_names)):
        if idx in box.ap_class_index:
            p, r, ap50, ap = box.class_result(list(box.ap_class_index).index(idx))
            per_class.append({"class": class_names[idx], "precision": float(p), "recall": float(r),
                               "ap50": float(ap50), "ap50_95": float(ap)})
        else:
            per_class.append({"class": class_names[idx], "precision": None, "recall": None,
                               "ap50": None, "ap50_95": None, "note": "no test-set instances of this class"})

    metrics = {
        "country": country,
        "model": args.model,
        "architecture": "transformer (RT-DETR)",
        "seed": args.seed,
        "epochs": args.epochs,
        "imgsz": args.imgsz,
        "batch": args.batch,
        "train_seconds": round(train_seconds, 1),
        "split_config": split_config,
        "overall": {
            "precision_mean": float(box.mp),
            "recall_mean": float(box.mr),
            "map50": float(box.map50),
            "map50_95": float(box.map),
            "map75": float(box.map75),
        },
        "per_class": per_class,
        "train_run_dir": str(Path(args.project) / run_name),
        "val_run_dir": str(run_dir),
    }

    metrics_path = run_dir / "transformer_metrics.json"
    metrics_path.write_text(json.dumps(metrics, indent=2) + "\n")

    report_lines = [
        f"# Phase 4 transformer (RT-DETR) baseline - {country}",
        "",
        f"Generated by `scripts/train_transformer_baseline.py`. Same {country} train/val/test split "
        f"as phase 2's CNN baseline (`data/splits/{country}/`, seed={split_config['seed'] if split_config else args.seed}) - "
        f"only the model architecture changed (YOLO11n -> RT-DETR), so any difference below is "
        f"attributable to architecture, not data.",
        "",
        "## Config (for reproducibility)",
        "",
        f"- model: `{args.model}`",
        f"- seed: {args.seed}",
        f"- epochs: {args.epochs} (patience={args.patience})",
        f"- imgsz: {args.imgsz}, batch: {args.batch}",
        f"- train time: {train_seconds / 60:.1f} min",
    ]
    if split_config:
        report_lines += [
            f"- split: train={split_config['n_train']} val={split_config['n_val']} "
            f"test={split_config['n_test']} out of {split_config['n_total']} total "
            f"(seed={split_config['seed']}, fractions "
            f"{split_config['train_frac']}/{split_config['val_frac']}/{split_config['test_frac']})",
        ]
    report_lines += [
        "",
        "## Overall metrics (on the held-out TEST split)",
        "",
        "| metric | value |",
        "|---|---|",
        f"| mAP@50 | {box.map50:.4f} |",
        f"| mAP@50-95 | {box.map:.4f} |",
        f"| mAP@75 | {box.map75:.4f} |",
        f"| mean precision | {box.mp:.4f} |",
        f"| mean recall | {box.mr:.4f} |",
    ]

    if cnn_metrics is not None:
        cnn_overall = cnn_metrics.get("overall", {})
        report_lines += [
            "",
            "## Architecture comparison: RT-DETR (transformer) vs. YOLO11n (CNN, phase 2)",
            "",
            f"Same {country} split, same seed, same held-out test images - the only thing that "
            f"changed between this run and phase 2's is the model architecture "
            f"(`{cnn_metrics.get('model', 'yolo11n.pt')}` -> `{args.model}`).",
            "",
            "| metric | YOLO11n (CNN) | RT-DETR (transformer) | Δ (transformer - CNN) |",
            "|---|---|---|---|",
        ]
        for key, label in [("map50", "mAP@50"), ("map50_95", "mAP@50-95"), ("map75", "mAP@75"),
                            ("precision_mean", "mean precision"), ("recall_mean", "mean recall")]:
            cnn_v = cnn_overall.get(key)
            rtdetr_v = metrics["overall"][key]
            if cnn_v is None:
                report_lines.append(f"| {label} | - | {rtdetr_v:.4f} | - |")
            else:
                report_lines.append(f"| {label} | {cnn_v:.4f} | {rtdetr_v:.4f} | {rtdetr_v - cnn_v:+.4f} |")
        cnn_train_s = cnn_metrics.get("train_seconds")
        if cnn_train_s:
            report_lines += [
                "",
                f"Training time: YOLO11n {cnn_train_s / 60:.1f} min vs. RT-DETR {train_seconds / 60:.1f} min "
                f"({train_seconds / cnn_train_s:.1f}x).",
            ]
    else:
        report_lines += [
            "",
            "## Architecture comparison",
            "",
            f"No CNN comparison file found at `{args.compare_against}` - run phase 2's "
            f"`train_cnn_baseline.py` first (or pass `--compare-against`) to get a side-by-side table.",
        ]

    report_lines += [
        "",
        "## Per-class precision / recall / AP (on the held-out TEST split)",
        "",
        "| class | precision | recall | AP50 | AP50-95 |",
        "|---|---|---|---|---|",
    ]
    for c in per_class:
        if c["precision"] is None:
            report_lines.append(f"| {c['class']} | - | - | - | - ({c.get('note', '')}) |")
        else:
            report_lines.append(f"| {c['class']} | {c['precision']:.4f} | {c['recall']:.4f} | "
                                 f"{c['ap50']:.4f} | {c['ap50_95']:.4f} |")
    train_run_dir = Path(args.project) / run_name
    report_lines += [
        "",
        "## Artifacts",
        "",
        f"- confusion matrix: `{run_dir / 'confusion_matrix.png'}` (raw counts) and "
        f"`{run_dir / 'confusion_matrix_normalized.png'}`",
        f"- PR / F1 / precision / recall curves: `{run_dir}/Box{{PR,F1,P,R}}_curve.png`",
        f"- training curves (loss/mAP per epoch): `{train_run_dir / 'results.png'}` "
        f"(from the `train` run at `{train_run_dir}`, not this `val` run directory)",
        f"- machine-readable metrics: `{metrics_path}`",
        "",
    ]
    report_path = run_dir / "transformer_report.md"
    report_path.write_text("\n".join(report_lines) + "\n")

    print(f"\nWrote {metrics_path}")
    print(f"Wrote {report_path}")
    print(f"\nOverall: mAP50={box.map50:.4f} mAP50-95={box.map:.4f} "
          f"(precision={box.mp:.4f}, recall={box.mr:.4f})")
    if cnn_metrics is not None:
        print(f"vs. CNN (phase 2): mAP50={cnn_metrics['overall']['map50']:.4f} "
              f"(Δ={box.map50 - cnn_metrics['overall']['map50']:+.4f})")
    if any(c["precision"] is None for c in per_class):
        missing = [c["class"] for c in per_class if c["precision"] is None]
        print(f"\nNOTE: {missing} had no instances in the test split - their AP is undefined, not "
              f"zero.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


写入phase 2实际跑出来的`baseline_metrics.json`(CNN/YOLO11n的数字,mAP@50=0.3113)到`train_transformer_baseline.py`默认读取的路径——这样第5步不需要在Kaggle上重新训练CNN baseline就能生成架构对比表。

In [ ]:
%%writefile runs/detect/val/baseline_metrics.json
{
  "country": "Czech",
  "model": "yolo11n.pt",
  "seed": 42,
  "epochs": 100,
  "imgsz": 640,
  "batch": 16,
  "train_seconds": 804.2,
  "split_config": {
    "country": "Czech",
    "seed": 42,
    "train_frac": 0.7,
    "val_frac": 0.15,
    "test_frac": 0.15,
    "n_total": 1072,
    "n_train": 750,
    "n_val": 160,
    "n_test": 162
  },
  "overall": {
    "precision_mean": 0.36503641412516236,
    "recall_mean": 0.3562095312095312,
    "map50": 0.31131808861114707,
    "map50_95": 0.10698202799119645,
    "map75": 0.03703188833322669
  },
  "per_class": [
    {
      "class": "longitudinal_crack",
      "precision": 0.5047194543699662,
      "recall": 0.48951048951048953,
      "ap50": 0.45195569558861326,
      "ap50_95": 0.16245754807785573
    },
    {
      "class": "transverse_crack",
      "precision": 0.28914457410753075,
      "recall": 0.26153846153846155,
      "ap50": 0.27879348160255923,
      "ap50_95": 0.08986405997233068
    },
    {
      "class": "alligator_crack",
      "precision": 0.3861548420557492,
      "recall": 0.48148148148148145,
      "ap50": 0.32401956834176315,
      "ap50_95": 0.1019093969505072
    },
    {
      "class": "pothole",
      "precision": 0.2801267859674032,
      "recall": 0.19230769230769232,
      "ap50": 0.19050360891165283,
      "ap50_95": 0.07369710696409225
    }
  ],
  "train_run_dir": "/kaggle/working/runs/phase2/czech_baseline",
  "val_run_dir": "/kaggle/working/runs/detect/val"
}


## 1. 安装依赖

In [ ]:
!pip install -q ultralytics pandas Pillow pyyaml
import torch
print("CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (go enable the GPU accelerator in Session options!)")


## 2. 下载 + MD5校验

优先用上面挂载的Dataset缓存,没有才走FigShare(和phase 3一样)。

In [ ]:
!python scripts/download_rdd2022.py --skip-extract

## 3. 解压转换Czech

只解压转换Czech一个国家——和phase 2一样,`--countries Czech`。

In [ ]:
!python scripts/extract_convert_per_country.py --countries Czech

## 4. 重新生成Czech split + 哈希校验

用和phase 2完全相同的命令(`--seed 42`)重新生成split。因为底层的Czech原始数据、提取逻辑、`random.Random(seed).shuffle()`都没变,这次生成的train/val/test应该和phase 2当时逐张图片完全一致——下面这格用哈希校验来确认这一点,而不是assume。如果校验失败会直接报错停在这里,不会带着"可能不是同一份数据"这个疑点继续往下训练。

In [ ]:
!python scripts/make_country_split.py --country Czech --seed 42

In [ ]:
import hashlib
from pathlib import Path

# Fingerprints of phase 2's ACTUAL committed data/splits/Czech/{train,val,test}.txt.
#
# CANONICAL FORM - EXPECTED below MUST be computed the exact same way this cell computes it:
#     basenames = sorted(Path(line).name for line in text.splitlines() if line.strip())
#     md5("\n".join(basenames).encode()).hexdigest()
# The first phase 4 run failed here on CORRECT data because these constants had been computed with
# a `sed 's#.*/##' f.txt | sort | md5sum` shell pipeline instead - that stream ends with a TRAILING
# NEWLINE after the last basename, which "\n".join() does not produce, so the two methods can never
# agree even on byte-identical input. Recompute with the snippet above, never with the shell one.
EXPECTED = {
    "train": ("136cf985f729fa01564759a61ed7a0af", 750),
    "val": ("e467ab9b030cbbae0b8cdac889ad12e8", 160),
    "test": ("0f7598cb491646863cbdfee489c0b8a7", 162),
}

split_dir = Path("data/splits/Czech")
actual = {}
all_ok = True
for split_name, (expected_hash, expected_n) in EXPECTED.items():
    lines = (split_dir / f"{split_name}.txt").read_text().splitlines()
    basenames = sorted(Path(l).name for l in lines if l.strip())
    actual_hash = hashlib.md5("\n".join(basenames).encode()).hexdigest()
    actual[split_name] = basenames
    status = "OK" if (actual_hash == expected_hash and len(basenames) == expected_n) else "MISMATCH"
    if status != "OK":
        all_ok = False
    print(f"  {split_name}: n={len(basenames)} (expected {expected_n}), hash={actual_hash} "
          f"(expected {expected_hash}) -> {status}")

if not all_ok:
    # Print enough to diagnose the cause WITHOUT spending another GPU run: if the counts are
    # 750/160/162 and the total is 1072, the image SET is correct and only the ordering differs
    # (a split-algorithm/environment issue); if the total differs, extract_convert_per_country.py
    # produced a different set of Czech images than phase 2 did (a data-pipeline issue).
    print("\n--- diagnostics ---")
    for split_name, basenames in actual.items():
        print(f"{split_name}: n={len(basenames)} first={basenames[0]} last={basenames[-1]}")
    print(f"total across splits: {sum(len(v) for v in actual.values())} (phase 2 had 1072)")
    print("phase 2 reference: train first=Czech__Czech_000010.jpg last=Czech__Czech_003589.jpg | "
          "val first=Czech__Czech_000006.jpg last=Czech__Czech_003584.jpg | "
          "test first=Czech__Czech_000061.jpg last=Czech__Czech_003571.jpg")
    raise RuntimeError(
        "Czech split does NOT match phase 2's committed split - STOPPING before training. "
        "This would not be a clean architecture-only comparison. Compare the diagnostics above "
        "against the phase 2 reference line to see whether the image set or only the ordering "
        "changed."
    )
print("\nAll three splits match phase 2's committed split exactly - safe to train RT-DETR on this data.")


## 5. 训练RT-DETR-L + 评测 + 架构对比

**训练时间预估**:RT-DETR-L(3280万参数)比YOLO11n(260万参数)大一个数量级,`train_transformer_baseline.py`默认把batch从phase 2的16降到8留显存余量。phase 2跑750张图、100 epoch花了13.4分钟;RT-DETR大概率会明显更慢,具体多少这次是第一次实测。`patience=20`同样会在没提升时提前停。

训练完在**同一个held-out test split**(和phase 2完全一样的162张图)上评测,脚本会自动读取上面写入的phase 2 CNN数字,生成一张YOLO11n vs. RT-DETR的架构对比表(mAP@50/50-95/75、precision、recall,以及训练耗时对比)。

In [ ]:
!python scripts/train_transformer_baseline.py \
    --data data/splits/Czech/dataset.yaml \
    --model rtdetr-l.pt \
    --epochs 100 \
    --imgsz 640 \
    --batch 8 \
    --seed 42


## 6. 核对结果

In [ ]:
!echo '--- split_config.json ---'
!cat data/splits/Czech/split_config.json
!echo
!echo '--- transformer_report.md ---'
!find runs/phase4 -name transformer_report.md -exec cat {} \;
!echo
!echo '--- 产出文件树 ---'
!find runs/phase4 data/splits/Czech -type f | sort


## 7. 收尾:打包成一个zip一次性下载

把这次regenerate过的(已校验通过的)split文件 + phase 4的训练/评测产出一起打包。

In [ ]:
!zip -r phase4_deliverables.zip \
    data/splits/Czech \
    runs/phase4 \
    -x "*.cache"
!ls -lh phase4_deliverables.zip


下载好`phase4_deliverables.zip`后告诉我一声,我来同步到Mac仓库、做校验、更新README。